In [19]:
import os
from dotenv import load_dotenv

os.environ['OPEN_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_TRACING_V2'] = "true"
os.environ['LANGCHAIN_PROJECT'] = os.getenv("LANGCHAIN_PROJECT")

In [20]:
##From the website, we need to scrape the data
## Web Based Loader
from langchain_community.document_loaders import WebBaseLoader
import bs4
web_load = WebBaseLoader(web_paths=("https://docs.langchain.com/langsmith/administration-overview",))
docs = web_load.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='Overview - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationAccount administrationOverviewGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusOn this pageResource hierarchyOrganizationsWorkspacesApplicationsResourcesAdditional infoResource tagsUser management and RBACUsersAPI keysExpiration datesPer

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
documents = text_splitter.split_documents(docs)
documents

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='Overview - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationAccount administrationOverviewGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusOn this pageResource hierarchyOrganizationsWorkspacesApplicationsResourcesAdditional infoResource tagsUser management and RBACUsersAPI keysExpiration datesPer

In [22]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [23]:
from langchain_community.vectorstores import FAISS
vectorstore_db = FAISS.from_documents(documents,embeddings)

In [24]:
query = "LangSmith resource tags"
result = vectorstore_db.similarity_search(query)
result[0].page_content

'* Data retention settings and usage limits will be available soon for the organization level as well ** Self-hosted installations may enable workspace-level invites of users to the organization via a feature flag. See the self-hosted user management docs for details.\n\u200bResource tags\nResource tags allow you to further segregate resources within a workspace for use with ABAC. Each tag is a key-value pair that you can assign to a resource.\nLangSmith resource tags are very similar to tags in cloud services like AWS.\nNavigate to Settings in the LangSmith UI to select the Resource tags page in the sidebar.\n\u200bUser management and RBAC\n\u200bUsers\nA user is a person who has access to LangSmith. Users can be members of one or more organizations and workspaces within those organizations.\nOrganization members are managed on the Settings page under Members and roles.\nAnd workspace members are managed on the Workspaces page under Settings.\n\u200bAPI keys'

In [31]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model = "gpt-4o")

prompt = ChatPromptTemplate.from_template(
   """
Answer the following question based only on provided context:
<context>
{context}
</context>
 
   """

)

document_chain = create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on provided context:\n<context>\n{context}\n</context>\n\n   '), additional_kwargs={})])
| ChatOpenAI(profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temp

In [32]:
from langchain_core.documents import Document
document_chain.invoke({
    "input" : "LangSmith resource tags",
     "context" : [Document(page_content= "Resource tags allow you to further segregate resources within a workspace for use with ABAC. Each tag is a key-value pair that you can assign to a resource.LangSmith resource tags are very similar to tags in cloud services like AWS.Navigate to Settings in the LangSmith UI to select the Resource tags page in the sidebar.")]
})

'Resource tags in LangSmith are key-value pairs that you can assign to a resource to further segregate resources within a workspace for use with Attribute-Based Access Control (ABAC). You can manage these tags by navigating to the Settings in the LangSmith user interface and selecting the Resource tags page from the sidebar. This functionality is similar to how tags are used in cloud services like AWS.'

In [34]:
## Input--->Retriever--->vectorstoredb
retriever = vectorstore_db.as_retriever()

In [40]:
from langchain_classic.chains import create_retrieval_chain
retrieval_chain = create_retrieval_chain(retriever,document_chain)
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000244CFA88B50>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on provided context:\n<context>\n{context}\n</context>\n\n   '), additional_kwargs={})])
            | Cha

In [43]:
## GET the response from LLM
response = retrieval_chain.invoke({"input": "LangSmith resource tags"})
response

{'input': 'LangSmith resource tags',
 'context': [Document(id='0dbaf919-af0f-4022-b812-c0512d7d9ba5', metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='* Data retention settings and usage limits will be available soon for the organization level as well ** Self-hosted installations may enable workspace-level invites of users to the organization via a feature flag. See the self-hosted user management docs for details.\n\u200bResource tags\nResource tags allow you to further segregate resources within a workspace for use with ABAC. Each tag is a key-value pair that you can assign to a resource.\nLangSmith resource tags are very similar to tags in cloud services like AWS.\nNavigate to Settings in the LangSmith UI to select the Resource tags page in the sidebar.\n\u200bUser management and RBAC\n\u200bUsers\nA user is a person who has access to LangSmith. Users can be members of one or